# 🔐 Notebook 2: CRC32 vs MD5 vs SHA-256 vs BLAKE2b

In Notebook 1 we saw silent corruption. Now we fix it by storing a **checksum** next to the data and recomputing it on read.

Not all checksums are the same. They trade **speed** against **strength**:

| Algorithm | Size | Designed for | Collision-resistant vs attacker? |
|---|---|---|---|
| **CRC32** | 4 bytes | accidental bit errors (disk, Ethernet frame) | ❌ no |
| **MD5** | 16 bytes | fast content fingerprint | ❌ broken since 2004 |
| **SHA-256** | 32 bytes | cryptographic integrity, signatures | ✅ yes |
| **BLAKE2b** | 32 bytes (tunable) | modern SHA-2 alternative | ✅ yes, and faster |

Rule of thumb:
- You only worry about **random bit flips** → CRC32 is plenty and basically free.
- You need to **detect tampering** by someone malicious → you need a cryptographic hash (SHA-256 / BLAKE2b), ideally with a secret key (HMAC — we'll see that in Notebook 3).

Big real-world storage systems use *both*: CRC32 per disk page to catch hardware faults cheaply, plus a cryptographic hash at the object level for end-to-end integrity.

## 🛠️ Setup

```bash
cd 02-distributed-primitives/checksum
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.

Everything here uses only the Python standard library (`zlib`, `hashlib`). No installs required beyond `uv sync`.

## 🛠️ Helpers: store `[len(digest)][digest][payload]` on disk

A very common on-disk layout for a checksummed record:

```
┌──────────┬───────────────────┬──────────────────────────────┐
│ len  (2B)│ digest (N bytes)  │  payload (the actual bytes)  │
└──────────┴───────────────────┴──────────────────────────────┘
```

Real systems (Kafka, SQLite, Postgres pages, Parquet, ...) use variations of this idea — checksum + length + payload.

In [1]:
import zlib, hashlib, time, os, tempfile

def crc32(b):    return zlib.crc32(b).to_bytes(4, 'big')
def md5(b):      return hashlib.md5(b).digest()
def sha256(b):   return hashlib.sha256(b).digest()
def blake2b(b):  return hashlib.blake2b(b, digest_size=32).digest()

ALGS = {'crc32': crc32, 'md5': md5, 'sha256': sha256, 'blake2b': blake2b}

def store(path, payload, alg):
    digest = ALGS[alg](payload)
    with open(path, 'wb') as f:
        f.write(len(digest).to_bytes(2, 'big'))
        f.write(digest)
        f.write(payload)

def load(path, alg):
    with open(path, 'rb') as f:
        n = int.from_bytes(f.read(2), 'big')
        stored_digest = f.read(n)
        payload = f.read()
    if ALGS[alg](payload) != stored_digest:
        raise ValueError(f'checksum mismatch ({alg}) — file is corrupt')
    return payload


## ✅ Round-trip + corruption test

For each algorithm:
1. Store some bytes with the checksum.
2. Load them back — should succeed.
3. Flip a single byte in the file, try again — should be detected.

In [2]:
WORKDIR = tempfile.mkdtemp(prefix='chk_')
data = b'hello world ' * 100

for alg in ALGS:
    path = os.path.join(WORKDIR, f'data.{alg}')
    store(path, data, alg)
    ok = load(path, alg) == data
    print(f'{alg:8s} round-trip ok? {ok}')

    # Corrupt one byte in the payload and try to read again.
    raw = bytearray(open(path, 'rb').read())
    raw[-1] ^= 0xFF
    open(path, 'wb').write(raw)

    try:
        load(path, alg)
        print(f'         ❌ corruption NOT detected')
    except ValueError as e:
        print(f'         ✅ corruption detected: {e}')


crc32    round-trip ok? True
         ✅ corruption detected: checksum mismatch (crc32) — file is corrupt
md5      round-trip ok? True
         ✅ corruption detected: checksum mismatch (md5) — file is corrupt
sha256   round-trip ok? True
         ✅ corruption detected: checksum mismatch (sha256) — file is corrupt
blake2b  round-trip ok? True
         ✅ corruption detected: checksum mismatch (blake2b) — file is corrupt


## 🌊 Streaming: checksumming files you can't fit in RAM

Everything above reads the whole payload into memory. For a 50 GB backup file or an HTTP upload that would be a disaster. The trick: **feed the data to the hasher in chunks** — every hash in Python supports `.update(chunk)` and gives you the same digest at the end.

This is how `s3 cp`, `rsync`, `sha256sum`, and Git compute hashes for huge files.

In [3]:
CHUNK = 64 * 1024  # 64 KB at a time

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        while True:
            chunk = f.read(CHUNK)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

# Make a 20 MB file of random bytes
big = os.path.join(WORKDIR, 'big.bin')
with open(big, 'wb') as f:
    f.write(os.urandom(20 * 1024 * 1024))

print('streaming sha256:', sha256_file(big)[:16], '...')
# Sanity check — same result as hashing the whole thing at once
whole = hashlib.sha256(open(big, 'rb').read()).hexdigest()
print('whole-file sha256:', whole[:16], '...')
print('match?', sha256_file(big) == whole)


streaming sha256: 942b5e6ae5cefde2 ...
whole-file sha256: 942b5e6ae5cefde2 ...
match? True


## ⚡ Performance comparison on 20 MB

Numbers will vary by CPU. What matters is the **shape**: CRC32 is typically the cheapest, MD5 the cheapest cryptographic-ish option, SHA-256 stronger but slower, BLAKE2b usually beats SHA-256 in pure software.

In [4]:
blob = os.urandom(20 * 1024 * 1024)

print(f'{"algorithm":10s} {"ms/20MB":>10s} {"MB/s":>10s}')
print('-' * 32)
for name, fn in ALGS.items():
    runs = 5
    t0 = time.perf_counter()
    for _ in range(runs):
        fn(blob)
    ms = (time.perf_counter() - t0) * 1000 / runs
    mb_s = 20 / (ms / 1000)
    print(f'{name:10s} {ms:10.2f} {mb_s:10.0f}')


algorithm     ms/20MB       MB/s


--------------------------------
crc32            0.47      42788
md5             22.29        897


sha256           6.36       3145


blake2b         13.06       1532


## 📊 Choosing: real-world examples

| System | What they use | Why |
|---|---|---|
| **TCP / IP header** | 16-bit ones'-complement checksum | catches common bit flips cheaply in hardware |
| **Ethernet frames** | CRC32 | hardware-cheap, catches all 1-3 bit burst errors |
| **ZFS / Btrfs** | Fletcher / CRC32C + optional SHA-256 | per-block bit-rot detection + scrub |
| **Postgres pages** | CRC32C (when `data_checksums` is on) | detect torn/corrupt 8 KB pages |
| **Kafka records** | CRC32C per record batch | reject corrupt batches on disk / on wire |
| **Git** | SHA-1 (→ SHA-256) | content-addressed storage; objects named by their hash |
| **S3 ETag** | MD5 of object (or composite for multipart) | client can verify download |
| **HTTPS / TLS** | HMAC-SHA256 or AEAD (GCM) | integrity **and** authenticity of every packet |
| **Signed URLs / JWT** | HMAC-SHA256 | server signs, nobody else can forge |

Most production storage layers combine **cheap per-block CRC** (for hardware faults) with **cryptographic hash at the object level** (for end-to-end integrity and dedup).

👉 Notebook 3 shows the two patterns you need on top of hashes: **ETag** (for HTTP conditional requests) and **HMAC** (for tamper detection with a secret key).